# OpenImages ResNet101

- https://github.com/openimages/dataset/blob/main/tools/classify_oidv2.py

In [1]:
# download model & necessary data
!wget https://storage.googleapis.com/openimages/2017_07/classes-trainable.txt
!wget https://storage.googleapis.com/openimages/2017_07/class-descriptions.csv
!wget https://storage.googleapis.com/openimages/2017_07/oidv2-resnet_v1_101.ckpt.tar.gz
!wget https://raw.githubusercontent.com/openimages/dataset/master/tools/classify_oidv2.py
!tar -xzf oidv2-resnet_v1_101.ckpt.tar.gz

!wget -O cat.jpg https://farm6.staticflickr.com/5470/9372235876_d7d69f1790_b.jpg

--2026-01-05 12:36:58--  https://storage.googleapis.com/openimages/2017_07/classes-trainable.txt
Resolving storage.googleapis.com (storage.googleapis.com)... 142.251.108.207, 142.251.121.207, 192.178.142.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.251.108.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 49202 (48K) [text/plain]
Saving to: ‘classes-trainable.txt.1’

classes-trainable.t 100%[===================>]  48.05K  --.-KB/s    in 0.001s  

2026-01-05 12:36:58 (93.6 MB/s) - ‘classes-trainable.txt.1’ saved [49202/49202]

--2026-01-05 12:36:58--  https://storage.googleapis.com/openimages/2017_07/class-descriptions.csv
Resolving storage.googleapis.com (storage.googleapis.com)... 142.251.108.207, 142.251.121.207, 192.178.142.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.251.108.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 452866 (442K) [text/csv]
Saving to: ‘class-d

In [2]:
import tensorflow as tf

In [3]:
def LoadLabelMap(labelmap_path, dict_path):
    """Load index->mid and mid->display name maps.

    Args:
        labelmap_path: path to the file with the list of mids, describing
            predictions.
        dict_path: path to the dict.csv that translates from mids to display names.

    Returns:
        labelmap: an index to mid list
        label_dict: mid to display name dictionary
    """
    labelmap = [line.rstrip() for line in tf.compat.v1.gfile.GFile(labelmap_path)]

    label_dict = {}
    for line in tf.compat.v1.gfile.GFile(dict_path):
        words = [word.strip(' "\n') for word in line.split(',', 1)]
        label_dict[words[0]] = words[1]

    return labelmap, label_dict

In [4]:
images = ["cat.jpg"]
checkpoint_path = 'oidv2-resnet_v1_101.ckpt'
top_k = 10
score_threshold = 0.3

In [5]:
# Load labelmap and dictionary from disk.
labelmap, label_dict = LoadLabelMap('classes-trainable.txt', 'class-descriptions.csv')

In [6]:
g = tf.compat.v1.Graph()
with g.as_default():
    with tf.compat.v1.Session() as sess:
        saver = tf.compat.v1.train.import_meta_graph(f'{checkpoint_path}.meta')
        saver.restore(sess, checkpoint_path)

        input_values = g.get_tensor_by_name('input_values:0')
        predictions = g.get_tensor_by_name('multi_predictions:0')

        for image_filename in images:
            compressed_image = tf.compat.v1.gfile.FastGFile(image_filename, 'rb').read()
            predictions_eval = sess.run(
                predictions, feed_dict={input_values: [compressed_image]}
            )

            pred = predictions_eval.argsort()[::-1]  # indices sorted by score
            if top_k > 0:
                pred = pred[:top_k]
            if score_threshold is not None:
                pred = [i for i in pred
                        if predictions_eval[i] >= score_threshold]

            print(f'Image: {image_filename}')
            for idx in pred:
                mid = labelmap[idx]
                display_name = label_dict[mid]
                score = predictions_eval[idx]
                print(f'{idx:04d}: {mid} - {display_name} (score = {score:.2f})')

Instructions for updating:
Use tf.gfile.GFile.


Image: cat.jpg
3272: /m/068hy - Pet (score = 0.96)
1076: /m/01yrx - Cat (score = 0.96)
0708: /m/01l7qd - Whiskers (score = 0.91)
4755: /m/0jbk - Animal (score = 0.90)
2847: /m/04rky - Mammal (score = 0.89)
2036: /m/0307l - Felidae (score = 0.79)
3574: /m/07k6w8 - Small to medium-sized cats (score = 0.78)
4799: /m/0k0pj - Nose (score = 0.70)
1495: /m/02cqfm - Close-up (score = 0.58)
0036: /m/012c9l - Domestic short-haired cat (score = 0.40)
